In [ ]:
!pip install kagglehub

import kagglehub
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

path = kagglehub.dataset_download(
    "kaushil268/disease-prediction-using-machine-learning"
)

print("Dataset path:", path)


train_df = pd.read_csv(os.path.join(path, "Training.csv"))
test_df = pd.read_csv(os.path.join(path, "Testing.csv"))

target = "prognosis"

X_train = train_df.drop(target, axis=1)
y_train = train_df[target]

X_test = test_df.drop(target, axis=1)
y_test = test_df[target]

X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)
X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train = X_train / (X_train.max() + 1e-8)
X_test = X_test / (X_test.max() + 1e-8)


le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

num_classes = len(le.classes_)
print("Classes:", num_classes)

X_train = torch.tensor(X_train.values).float()
X_test = torch.tensor(X_test.values).float()

y_train = torch.tensor(y_train).long()
y_test = torch.tensor(y_test).long()

class AECA_Model(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = AECA_Model(X_train.shape[1], num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0003)

epochs = 300

for epoch in range(epochs):
    model.train()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

model.eval()

with torch.no_grad():
    preds = model(X_test)
    probs = torch.softmax(preds, dim=1)

    predicted = torch.argmax(probs, dim=1)
    accuracy = (predicted == y_test).float().mean()

print("\nAccuracy:", accuracy.item())

symptom_list = list(train_df.drop("prognosis", axis=1).columns)

print("\nSample symptoms:")
print(symptom_list[:25])

while True:
    user_input = input("\nEnter symptoms (comma-separated) or 'exit':\n")

    if user_input.lower() == "exit":
        break

    user_symptoms = [s.strip().lower() for s in user_input.split(",")]

    input_vector = [0] * len(symptom_list)

    matched = []

    for i, symptom in enumerate(symptom_list):
        for us in user_symptoms:
            if us in symptom.lower() or symptom.lower() in us:
                input_vector[i] = 1
                matched.append(symptom)

    input_tensor = torch.tensor([input_vector]).float()

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)[0]

        top3 = torch.topk(probs, 3)

        print("\nTop Predictions:")

        for i in range(3):
            idx = top3.indices[i].item()
            conf = top3.values[i].item()
            disease = le.inverse_transform([idx])[0]

            print(f"{i+1}. {disease} ({conf:.3f})")

        top_conf = top3.values[0].item()

        if top_conf > 0.75:
            triage = "🔴 Critical"
        elif top_conf > 0.45:
            triage = "🟡 Urgent"
        else:
            triage = "🟢 Non-Urgent"

        print("\n🏥 Triage Level:", triage)

    print("\nMatched symptoms:", matched)


Using Colab cache for faster access to the 'disease-prediction-using-machine-learning' dataset.
Dataset path: /kaggle/input/disease-prediction-using-machine-learning
Classes: 41
Epoch 0, Loss: 3.737126111984253
Epoch 20, Loss: 2.4338958263397217
Epoch 40, Loss: 1.3724480867385864
Epoch 60, Loss: 0.5867393016815186
Epoch 80, Loss: 0.2187943309545517
Epoch 100, Loss: 0.10142023116350174
Epoch 120, Loss: 0.06080646067857742
Epoch 140, Loss: 0.042021602392196655
Epoch 160, Loss: 0.031377170234918594
Epoch 180, Loss: 0.02457388862967491
Epoch 200, Loss: 0.019884347915649414
Epoch 220, Loss: 0.01648862473666668
Epoch 240, Loss: 0.013928934000432491
Epoch 260, Loss: 0.011952389031648636
Epoch 280, Loss: 0.010389711707830429

Accuracy: 1.0

Sample symptoms:
['itching', 'skin_rash', 'nodal_skin_eruptions', 'continuous_sneezing', 'shivering', 'chills', 'joint_pain', 'stomach_pain', 'acidity', 'ulcers_on_tongue', 'muscle_wasting', 'vomiting', 'burning_micturition', 'spotting_ urination', 'fatigue